In [1]:
import pandas as pd

In [2]:
ttc = pd.read_csv(r'D:\Portfolio Projects\Public Transit Ridership\Data\RAW_Data\TTC Average Weekday Ridership(2007-2025).csv')
ttc.head()

,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,2007 Actual,1400000,1486000,1485000,1491000,1474000,1473000,1391000,1356000,1549000,1531000,1538000,1396000
1,2008 Actual,1379000,1476000,1519000,1446000,1479000,1459000,1444000,1434000,1574000,1520000,1600000,1484000
2,2009 Actual,1448000,1536000,1487000,1499000,1517000,1467000,1435000,1328000,1586000,1578000,1568000,1424000
3,2010 Actual,1474000,1538000,1489000,1500000,1530000,1492000,1473000,1412000,1615000,1622000,1541000,1457000
4,2011 Actual,1540000,1595000,1568000,1539000,1602000,1579000,1528000,1449000,1689000,1678000,1639000,1583000


In [3]:
# STEP 1 — rename first column if needed
# change this only if your first column is not exactly called "Year"
ttc = ttc.rename(columns={"Year": "year_raw"})

# STEP 2 — keep only actual ridership rows if needed
ttc = ttc[ttc["year_raw"].str.contains("Actual", case=False, na=False)].copy()

# STEP 3 — extract numeric year
ttc["year"] = ttc["year_raw"].str.extract(r"(\d{4})").astype(int)

# STEP 4 — reshape from wide to long
month_cols = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

ttc_long = ttc.melt(
    id_vars=["year_raw", "year"],
    value_vars=month_cols,
    var_name="month_str",
    value_name="ridership"
)

# STEP 5 — convert month names to month numbers
month_map = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

ttc_long["month"] = ttc_long["month_str"].map(month_map)

# STEP 6 — build proper monthly date
ttc_long["date"] = pd.to_datetime(
    dict(year=ttc_long["year"], month=ttc_long["month"], day=1)
)

# STEP 7 — final cleaned table
ttc_clean = (
    ttc_long[["date", "year", "month", "ridership"]]
    .sort_values("date")
    .reset_index(drop=True)
)

display(ttc_clean.head(15))
display(ttc_clean.tail(15))

,date,year,month,ridership
0,2007-01-01,2007,1,1400000
1,2007-02-01,2007,2,1486000
2,2007-03-01,2007,3,1485000
3,2007-04-01,2007,4,1491000
4,2007-05-01,2007,5,1474000
5,2007-06-01,2007,6,1473000
6,2007-07-01,2007,7,1391000
7,2007-08-01,2007,8,1356000
8,2007-09-01,2007,9,1549000
9,2007-10-01,2007,10,1531000


,date,year,month,ridership
213,2024-10-01,2024,10,1389000
214,2024-11-01,2024,11,1397000
215,2024-12-01,2024,12,1061000
216,2025-01-01,2025,1,1250000
217,2025-02-01,2025,2,1169000
218,2025-03-01,2025,3,1385000
219,2025-04-01,2025,4,1299000
220,2025-05-01,2025,5,1308000
221,2025-06-01,2025,6,1289000
222,2025-07-01,2025,7,1276000


In [4]:
ttc_clean.dtypes

date         datetime64[ns]
year                  int64
month                 int64
ridership             int64
dtype: object

In [5]:
ttc_clean.to_csv(r"D:\Portfolio Projects\Public Transit Ridership\Data\Interim\ttc_ridership.csv", index=False)